# RAG 知识底座 · 验收 Notebook

> 本文件是**验收脚本**，不是教学 demo。
> 目的：逐条验证 README 的承诺是否成立，对应教程第四段「发布前的最终检查」。
>
> | 本 notebook 的 cell | 验证什么 | 对应 README 段 |
> |---|---|---|
> | §1 环境自检 | Ollama / 模型 / 索引 / 评估集齐备 | 第 7 段 运行方式 |
> | §2 摄入侧 | chunk 元数据含 `source` / `chunk_id` | 第 2 段 数据来源与结构 |
> | §3 检索接口 | `retrieve()` 签名与返回结构 | 第 3 段 检索接口 |
> | §4 分数分布 | 为 `score_threshold` 定值提供数据 | 第 4 段 检索参数 |
> | §5 拒答策略 | 低分返回空列表 | 第 4 段 拒答策略 |
> | §6 评估证据 | `eval/results/summary.json` 存在且可读 | 第 5 段 评估证据 |
> | §7 汇总 | 全项 pass/fail 一览 | 教程第四段验收清单 |

**前置条件**：已跑过 `python -m rag.ingest` 和 `python -m rag.evaluate`。
本 notebook **不会**重新跑完整评估（耗时），只验收产出物。

> 所有检索、摄入、生成逻辑一律 `import` 现有模块，**不重写实现**——验收的对象就是真实资产。

## 0. 环境准备

自动定位项目根目录，把 `src/rag` 加入 `sys.path`，以兼容 `evaluate.py` 中 `from retrieve import retrieve` 的平铺导入风格。

In [31]:
import sys
import json
import inspect
import statistics
from pathlib import Path
from urllib.request import urlopen

# 向上查找项目根（含 src/ 的目录）
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAG_SRC = PROJECT_ROOT / 'src' / 'rag'
if str(RAG_SRC) not in sys.path:
    sys.path.insert(0, str(RAG_SRC))

print('PROJECT_ROOT :', PROJECT_ROOT)
print('RAG_SRC      :', RAG_SRC)
print('RAG_SRC 存在  :', RAG_SRC.exists())

PROJECT_ROOT : C:\Users\PC\rag-knowledge-base
RAG_SRC      : C:\Users\PC\rag-knowledge-base\src\rag
RAG_SRC 存在  : True


In [32]:
# 全局检查结果收集器，供 §7 汇总使用
CHECK_RESULTS = []

def record(name: str, passed: bool, detail: str = '') -> None:
    CHECK_RESULTS.append({'name': name, 'passed': passed, 'detail': detail})
    mark = '✅' if passed else '❌'
    print('{} {}{}'.format(mark, name, '  [{}]'.format(detail) if detail else ''))

print('检查结果收集器已就绪')

检查结果收集器已就绪


## 1. 环境自检

四项前置条件，任一缺失都会导致后续验收失败。每个失败项都会打印修复命令。

In [33]:
from src.rag.config import (
    OLLAMA_BASE_URL, EMBEDDING_MODEL, LLM_MODEL,
    VECTOR_STORE_DIR, RAW_DATA_DIR, BASE_DIR,
)

print('OLLAMA_BASE_URL :', OLLAMA_BASE_URL)
print('EMBEDDING_MODEL :', EMBEDDING_MODEL)
print('LLM_MODEL       :', LLM_MODEL)
print('VECTOR_STORE_DIR:', VECTOR_STORE_DIR)
print('RAW_DATA_DIR    :', RAW_DATA_DIR)

OLLAMA_BASE_URL : http://localhost:11434
EMBEDDING_MODEL : qwen3-embedding:0.6b
LLM_MODEL       : modelscope.cn/Qwen/Qwen2.5-3B-Instruct-GGUF:latest
VECTOR_STORE_DIR: C:\Users\PC\rag-knowledge-base\data\processed\chroma_db
RAW_DATA_DIR    : C:\Users\PC\rag-knowledge-base\data\raw


In [34]:
def _get_json(url: str, timeout: int = 5):
    with urlopen(url, timeout=timeout) as r:
        return json.loads(r.read().decode('utf-8'))


def _has_model(models: list, name: str) -> bool:
    """Ollama 模型名可能带 :latest 后缀，做宽松匹配。"""
    for m in models:
        if m == name or m.split(':')[0] == name.split(':')[0]:
            return True
    return False


# ---- 检查 1：Ollama 服务连通 ----
try:
    tags = _get_json(f'{OLLAMA_BASE_URL}/api/tags')
    ollama_models = [m['name'] for m in tags.get('models', [])]
    record('Ollama 服务连通', True, f'{len(ollama_models)} 个模型已拉取')
except Exception as e:
    ollama_models = []
    record('Ollama 服务连通', False, str(e))
    print('   修复: 启动 Ollama → ollama serve')

# ---- 检查 2：两个模型都已拉取 ----
emb_ok = _has_model(ollama_models, EMBEDDING_MODEL)
llm_ok = _has_model(ollama_models, LLM_MODEL)
record('Embedding 模型已拉取', emb_ok, EMBEDDING_MODEL)
record('LLM 模型已拉取', llm_ok, LLM_MODEL)
if not emb_ok:
    print(f'   修复: ollama pull {EMBEDDING_MODEL}')
if not llm_ok:
    print(f'   修复: ollama pull {LLM_MODEL}')

# ---- 检查 3：Chroma 索引已建立 ----
vs = Path(VECTOR_STORE_DIR)
vs_ok = vs.exists() and any(vs.iterdir())
record('向量库已建立', vs_ok, str(vs))
if not vs_ok:
    print('   修复: python -m rag.ingest')

# ---- 检查 4：评估集文件存在（兼容 .json / .jsonl） ----
candidates = [BASE_DIR / 'eval' / 'eval_set.json', BASE_DIR / 'eval' / 'eval_set.jsonl']
EVAL_SET_PATH = next((p for p in candidates if p.exists()), None)
record('评估集文件存在', EVAL_SET_PATH is not None, str(EVAL_SET_PATH))
if EVAL_SET_PATH is None:
    print('   修复: 确认 eval/eval_set.json 存在，且字段含 question / ground_truth')

✅ Ollama 服务连通  [3 个模型已拉取]
✅ Embedding 模型已拉取  [qwen3-embedding:0.6b]
✅ LLM 模型已拉取  [modelscope.cn/Qwen/Qwen2.5-3B-Instruct-GGUF:latest]
✅ 向量库已建立  [C:\Users\PC\rag-knowledge-base\data\processed\chroma_db]
✅ 评估集文件存在  [C:\Users\PC\rag-knowledge-base\eval\eval_set.jsonl]


## 2. 摄入侧验收（对应 README 第 2 段）

只对 `data/raw` 中第一个支持的文档做冒烟，**不写入 Chroma**，避免污染已有索引。

验证点：每个 chunk 的 `metadata` 必须包含 `source` 和 `chunk_id` —— 这是 Agent 溯源的最小信息集。

In [35]:
try:
    from src.rag.ingest import load_documents, split_documents

    raw_files = [p for p in Path(RAW_DATA_DIR).rglob('*')
                 if p.is_file() and p.suffix.lower() in {'.pdf', '.txt', '.md', '.markdown'}]

    if not raw_files:
        record('摄入侧冒烟', False, 'data/raw 下无支持的文件')
    else:
        sample = raw_files[0]
        # 只加载单个文件，避免全量读取
        single_dir_docs = load_documents(str(sample.parent))
        single_docs = [d for d in single_dir_docs if d.metadata.get('source') == sample.name]
        chunks = split_documents(single_docs)

        meta_ok = all('source' in c.metadata and 'chunk_id' in c.metadata for c in chunks)
        record('chunk 元数据含 source / chunk_id', meta_ok, f'{len(chunks)} 个 chunk')

        lens = [len(c.page_content) for c in chunks]
        if lens:
            print('   chunk 长度: min={} max={} 中位数={}'.format(
                min(lens), max(lens), int(statistics.median(lens))))
        print('   样例元数据:', json.dumps(chunks[0].metadata, ensure_ascii=False))
except Exception as e:
    record('摄入侧冒烟', False, str(e))
    print('   若报 persist_directory 类型错误，见 retrieve.py / ingest.py 修复说明')

正在扫描目录: C:\Users\PC\rag-knowledge-base\data\raw
  成功加载: knowledge_base.txt
共加载了 1 页/个文档。
正在进行文本切分 (Chunk Size: 300, Overlap: 30)...
切分完成，共生成 22 个 Chunk。
✅ chunk 元数据含 source / chunk_id  [22 个 chunk]
   chunk 长度: min=45 max=298 中位数=266
   样例元数据: {"source": "knowledge_base.txt", "chunk_id": "chunk_0000"}


## 3. 检索接口验收（对应 README 第 3 段）

**这是整个仓库最重要的契约。** 验证三件事：

1. `retrieve()` 能成功导入并调用
2. 签名与 README 示例一致（`query, top_k, score_threshold`）
3. 返回值每条包含 `text` / `source` / `score` / `metadata` 四个键

In [36]:
try:
    from src.rag.retrieve import retrieve

    sig = inspect.signature(retrieve)
    print('retrieve 签名:', sig)
    params = list(sig.parameters.keys())
    sig_ok = params[:3] == ['query', 'top_k', 'score_threshold']
    record('retrieve 签名符合契约', sig_ok, ', '.join(params))
except Exception as e:
    record('retrieve 导入', False, str(e))
    print('   若报 persist_directory 类型错误，修改 retrieve.py: persist_directory=str(VECTOR_STORE_DIR)')
    raise

retrieve 签名: (query: str, top_k: int = 3, score_threshold: float = 0.5) -> List[Dict[str, Any]]
✅ retrieve 签名符合契约  [query, top_k, score_threshold]


In [37]:
# 用 eval_001 的问题做真实调用
TEST_QUERY = 'URL命名应该用下划线还是短横线？'

try:
    hits = retrieve(TEST_QUERY, top_k=3, score_threshold=0.0)
    record('retrieve 调用成功', True, f'返回 {len(hits)} 条')

    if hits:
        required = {'text', 'source', 'score', 'metadata'}
        struct_ok = required.issubset(hits[0].keys())
        record('返回结构含 4 个必需键', struct_ok, ', '.join(sorted(hits[0].keys())))
        print()
        print('第一条完整返回（README 第 3 段应展示的形状）:')
        print(json.dumps(hits[0], ensure_ascii=False, indent=2)[:800])
    else:
        record('返回结构含 4 个必需键', False, '空列表，无法校验')
        print('   score_threshold=0.0 下仍返回空，说明索引为空或检索实现有问题')
except Exception as e:
    record('retrieve 调用成功', False, str(e))

✅ retrieve 调用成功  [返回 3 条]
✅ 返回结构含 4 个必需键  [metadata, score, source, text]

第一条完整返回（README 第 3 段应展示的形状）:
{
  "text": "## 2. URL设计规范\n\n### 2.1 基础格式\nhttps://api.example.com/{version}/{resource}/{id}/{sub-resource}\n\ntext\n\n### 2.2 命名规则\n- 全部使用小写字母\n- 多个单词之间使用短横线（-）连接，禁止使用下划线或驼峰\n- 正确示例：/user-profiles\n- 错误示例：/userProfiles、/user_profiles",
  "source": "knowledge_base.txt",
  "score": 0.585,
  "metadata": {
    "chunk_id": "chunk_0001",
    "source": "knowledge_base.txt"
  }
}


## 4. 检索分数分布（对应 README 第 4 段）

对评估集全部问题跑一遍 `score_threshold=0.0`，打印每题最高分。

**目的**：用数据回答「`score_threshold` 应该设多少」。`evaluate.py` 当前用的是 `0.3`；如果大量题目最高分低于 0.3，评估时 `contexts` 会全空，`ContextPrecision` / `ContextRecall` 将失去意义。

> README 第 4 段承诺的阈值，必须有这一节的分数分布作为依据，不能拍脑袋。

In [38]:
if EVAL_SET_PATH is None:
    print('评估集不存在，跳过')
else:
    eval_data = json.loads(EVAL_SET_PATH.read_text(encoding='utf-8'))
    print('评估集大小:', len(eval_data), '条\n')

    rows = []
    for item in eval_data:
        try:
            hs = retrieve(item['question'], top_k=3, score_threshold=0.0)
            top = hs[0]['score'] if hs else 0.0
        except Exception as e:
            top = -1.0
        rows.append((item.get('id', ''), top, item['question']))

    valid = [s for _, s, _ in rows if s >= 0]
    if valid:
        print('最高分分布: min={:.4f}  中位={:.4f}  max={:.4f}'.format(
            min(valid), statistics.median(valid), max(valid)))
    print()

    THRESHOLD = 0.3
    print('当前 evaluate.py 使用 score_threshold = {}'.format(THRESHOLD))
    below = sum(1 for s in valid if s < THRESHOLD)
    print('低于阈值的题数: {} / {}'.format(below, len(valid)))
    if below > 0:
        print('⚠️ 这些题的 contexts 在评估时会是空列表，相关指标不可信\n')
    else:
        print('✅ 全部题目都高于阈值\n')

    for qid, s, q in rows:
        flag = '⚠️' if 0 <= s < THRESHOLD else ('  ' if s >= 0 else '❌')
        print('{} {:<10} {:.4f}  {}'.format(flag, qid, s, q[:40]))

评估集大小: 12 条

最高分分布: min=0.5399  中位=0.5913  max=0.7408

当前 evaluate.py 使用 score_threshold = 0.3
低于阈值的题数: 0 / 12
✅ 全部题目都高于阈值

   eval_001   0.5850  URL命名应该用下划线还是短横线？
   eval_002   0.5975  DELETE请求应该返回什么状态码？
   eval_003   0.5451  API版本号应该放在哪里？
   eval_004   0.5714  分页参数page_size最大是多少？
   eval_005   0.6655  如何实现接口的幂等性？
   eval_006   0.5732  敏感信息在响应中应该如何处理？
   eval_007   0.5607  资源集合应该使用什么命名形式？
   eval_008   0.7408  批量删除应该使用什么HTTP方法和路径？
   eval_009   0.6613  API响应中的日期时间应该使用什么格式？
   eval_010   0.5399  枚举值应该使用字符串还是数字？
   eval_011   0.7237  分页列表接口的响应结构需要包含哪些分页字段？
   eval_012   0.6664  哪些API变更需要升级大版本？


## 5. 拒答策略验收（对应 README 第 4 段）

知识库外的问题，`retrieve()` 应返回**空列表**，而不是低分 chunk。
**返回空列表是预期行为，不是 bug** —— 上层 Agent 拿到空列表后自行降级（如转人工）。

In [39]:
OUT_OF_SCOPE = [
    '今天北京的天气怎么样？',
    '请帮我写一段快速排序的 Python 代码',
]

all_abstained = True
for q in OUT_OF_SCOPE:
    try:
        hs = retrieve(q, top_k=3)   # 用默认阈值
        abstained = len(hs) == 0
        all_abstained = all_abstained and abstained
        mark = '✅' if abstained else '⚠️'
        print('{} Q: {}'.format(mark, q))
        print('   -> {}'.format('返回空列表（正确拒答）' if abstained else f'意外命中 {len(hs)} 条'))
        if not abstained:
            print('   -> 最高分 {:.4f}，说明阈值偏低'.format(hs[0]['score']))
    except Exception as e:
        all_abstained = False
        print('❌ Q: {} -> {}'.format(q, e))

record('知识库外问题正确拒答', all_abstained)

✅ Q: 今天北京的天气怎么样？
   -> 返回空列表（正确拒答）
✅ Q: 请帮我写一段快速排序的 Python 代码
   -> 返回空列表（正确拒答）
✅ 知识库外问题正确拒答


## 6. 评估证据验收（对应 README 第 5 段）

README 第 5 段引用的每个数字，都要能在 `eval/results/` 中找到原始文件。
本节**不重新跑评估**，只验收 `evaluate.py` 的产出物是否存在且字段完整。

In [40]:
RESULTS_DIR = BASE_DIR / 'eval' / 'results'
summary_path = RESULTS_DIR / 'summary.json'
details_path = RESULTS_DIR / 'details.jsonl'

record('summary.json 存在', summary_path.exists(), str(summary_path))
record('details.jsonl 存在', details_path.exists(), str(details_path))

if not summary_path.exists():
    print('   修复: python -m rag.evaluate  (或 cd src/rag && python evaluate.py)')

✅ summary.json 存在  [C:\Users\PC\rag-knowledge-base\eval\results\summary.json]
✅ details.jsonl 存在  [C:\Users\PC\rag-knowledge-base\eval\results\details.jsonl]


In [41]:
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('=== summary.json ===')
    print(json.dumps(summary, ensure_ascii=False, indent=2))

    required_top = {'n_samples', 'llm_model', 'embedding_model', 'metrics'}
    required_metrics = {'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall'}

    top_ok = required_top.issubset(summary.keys())
    metrics_ok = required_metrics.issubset(summary.get('metrics', {}).keys())
    record('summary 含顶层必需字段', top_ok, ', '.join(summary.keys()))
    record('四个指标齐全', metrics_ok, ', '.join(summary.get('metrics', {}).keys()))
else:
    print('（summary.json 不存在，跳过字段校验）')

=== summary.json ===
{
  "n_samples": 1,
  "llm_model": "modelscope.cn/Qwen/Qwen2.5-3B-Instruct-GGUF:latest",
  "embedding_model": "qwen3-embedding:0.6b",
  "top_k": 3,
  "score_threshold": 0.3,
  "metrics": {
    "faithfulness": 1.0,
    "answer_relevancy": 0.8102,
    "context_precision": 1.0,
    "context_recall": 1.0
  }
}
✅ summary 含顶层必需字段  [n_samples, llm_model, embedding_model, top_k, score_threshold, metrics]
✅ 四个指标齐全  [faithfulness, answer_relevancy, context_precision, context_recall]


In [42]:
# 明细抽查：确认 details.jsonl 每条含 eval_id 与 source_section，便于逐题 trace
if details_path.exists():
    lines = [l for l in details_path.read_text(encoding='utf-8').splitlines() if l.strip()]
    print('details 条数:', len(lines))
    if lines:
        first = json.loads(lines[0])
        keys_ok = {'eval_id', 'source_section', 'user_input', 'response'}.issubset(first.keys())
        record('details 含 trace 字段', keys_ok, ', '.join(sorted(first.keys())[:6]))
        print('  首条 eval_id:', first.get('eval_id'))
        print('  首条 source_section:', first.get('source_section'))
else:
    print('（details.jsonl 不存在，跳过）')

details 条数: 1
✅ details 含 trace 字段  [answer_relevancy, context_precision, context_recall, contexts, eval_id, faithfulness]
  首条 eval_id: eval_001
  首条 source_section: 2.2 命名规则


## 7. 验收汇总

对照教程第四段「发布前的最终检查」，逐条打勾。
**只有全部 ✅ 才适合推送 / 对外发布 README。**

In [43]:
print('=' * 60)
print('验收汇总')
print('=' * 60)

passed = sum(1 for c in CHECK_RESULTS if c['passed'])
total = len(CHECK_RESULTS)

for c in CHECK_RESULTS:
    mark = '✅' if c['passed'] else '❌'
    print('{} {:<30} {}'.format(mark, c['name'], c['detail']))

print('-' * 60)
print('通过 {}/{}'.format(passed, total))

if passed == total:
    print('\n🎉 全部通过。可以推送并更新 README 第 3/4/5 段。')
else:
    print('\n⚠️ 存在未通过项，修复后再发布。')
    print('逐条对照上表，每项失败的修复命令见对应 cell。')

验收汇总
✅ Ollama 服务连通                    3 个模型已拉取
✅ Embedding 模型已拉取                qwen3-embedding:0.6b
✅ LLM 模型已拉取                      modelscope.cn/Qwen/Qwen2.5-3B-Instruct-GGUF:latest
✅ 向量库已建立                         C:\Users\PC\rag-knowledge-base\data\processed\chroma_db
✅ 评估集文件存在                        C:\Users\PC\rag-knowledge-base\eval\eval_set.jsonl
✅ chunk 元数据含 source / chunk_id   22 个 chunk
✅ retrieve 签名符合契约                query, top_k, score_threshold
✅ retrieve 调用成功                  返回 3 条
✅ 返回结构含 4 个必需键                   metadata, score, source, text
✅ 知识库外问题正确拒答                     
✅ summary.json 存在                C:\Users\PC\rag-knowledge-base\eval\results\summary.json
✅ details.jsonl 存在               C:\Users\PC\rag-knowledge-base\eval\results\details.jsonl
✅ summary 含顶层必需字段                n_samples, llm_model, embedding_model, top_k, score_threshold, metrics
✅ 四个指标齐全                         faithfulness, answer_relevancy, context_precision, context_recall
✅ details 含 tr

## 8. 手工核对清单（教程第四段原文）

以下几条无法自动断言，需要人眼确认：

- [ ] `README.md` 在 GitHub 上渲染正常，代码块、链接、目录树没有断裂
- [ ] `LICENSE`、`.gitignore`、`requirements.txt`、`.env.example` 四个文件齐备
- [ ] 仓库根目录没有意外提交的 `.env`、大体积二进制、虚拟环境文件夹
- [ ] README 第 3 段的 `retrieve()` 示例与本 notebook §3 的调用**逐字符一致**
- [ ] 仓库 Description 和 Topics 已填写，定位是「Agent 作品集知识底座」而非「问答 Demo」
- [ ] README 第 5 段的每个数字都能在 `eval/results/summary.json` 中找到对应值
- [ ] `docs/architecture.md` 里画清楚了「上层 Agent → retrieve() → 知识库」的调用关系

---

### 若 §4 显示阈值偏低 / 偏高

按以下顺序调整，**每次只改一个变量并重跑 `evaluate.py`**，否则数字不可比：

1. 先确定 `score_threshold`（用 §4 的分布中位数做参考）
2. 再考虑换 `EMBEDDING_MODEL`（如 `qwen3-embedding:0.6b` → `bge-m3`）
3. 最后调 `CHUNK_SIZE` / `CHUNK_OVERLAP`

每轮结果存一份 `eval/results/summary_<日期>.json`，README 只引用最新一份。